# Aggregation demo 

In [0]:
%run "../includes/configuration"

In [0]:
rece_results_df=spark.read.parquet(f'{presentation_folder_path}/race_results').where("race_year =2020")

In [0]:
display(rece_results_df)

In [0]:
from pyspark.sql.functions import count,countDistinct,sum,desc,asc,rank

## Count

In [0]:
rece_results_df.select(count("*")).show()


In [0]:
rece_results_df.select(count("driver_number")).show()


## CountDistinct

In [0]:
rece_results_df.select(countDistinct("driver_number")).show()

In [0]:
rece_results_df.select(countDistinct("race_name")).show()


## Sum

In [0]:
rece_results_df.select(sum("points")).show()

## Demo

In [0]:
rece_results_df.select(sum("points"),count("*"),countDistinct("race_name")).withColumnRenamed("sum(points)","total_points").withColumnRenamed("count(1)","total_races").withColumnRenamed("count(DISTINCT race_name)","total_circuits").show()

## groupBy

In [0]:
rece_results_df.groupBy("driver_name").sum("points").withColumnRenamed("sum(points)","total_points")


In [0]:
rece_results_df.groupBy("driver_name").agg(sum("points").alias("total_points"),countDistinct("race_name").alias("number_of_races")).show()



# Window functions

In [0]:
rece_results_df=spark.read.parquet(f'{presentation_folder_path}/race_results').where("race_year in(2019,2020)")

In [0]:
group_by_df=rece_results_df.groupBy("race_year","driver_name").agg(sum("points").alias("total_points"),countDistinct("race_name").alias("number_of_races")).orderBy(asc("race_year"),desc("total_points"))



In [0]:
display(group_by_df)

## rank 

In [0]:
from pyspark.sql.window import Window

In [0]:
driver_RankSpeck=Window.partitionBy("race_year").orderBy(desc("total_points"))

In [0]:
rank_df=group_by_df.withColumn("rank",rank().over(driver_RankSpeck))

In [0]:
display(rank_df)